# Notebook D: figures and tables for the paper and the slides

Run after Notebooks A, B and C. Nothing is trained here: every figure and table is drawn from the CSV files those
notebooks wrote to `report/`, so the paper and the slides always show the numbers in the repository.

| Output | Used in | Source |
|---|---|---|
| `report/paper/fig1_workflow.png` | paper Fig. 1 | study design (incl. PAMAP2 and the revision experiments) |
| `report/paper/fig2_windows_per_activity.png` | paper Fig. 2 | `table_q5.csv` |
| `report/paper/fig3_motion_heatmap.png` | paper Fig. 3 | `table_q2.csv` |
| `report/paper/fig4_people_cv.png` | paper Fig. 4 | `table_q3.csv` |
| `report/paper/fig5_masking_vs_retraining.png` | paper Fig. 5 | `table_rq3_roar.csv` (main split, k = 10) |
| `report/paper/fig6_positions_3models.png` | paper Fig. 6 | `table_rq3_positions.csv` (drawn by Notebook B) |
| `report/paper/tables_paper.tex` | paper Tables I to III | `table_rq2.csv`, `table_rq3_group_share.csv`, `table_rq3_methods.csv`, `table_external_check.csv` |
| `report/slides/*.png` | presentation | same CSV files, large fonts (at least 25 pt when the picture fills the slide width) |

In [1]:
import os, shutil, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Circle
from matplotlib.path import Path
os.makedirs('report/paper', exist_ok=True); os.makedirs('report/slides', exist_ok=True)
R = lambda f: pd.read_csv(os.path.join('report', f))
ACT = ['standing', 'lying', 'sitting', 'arm_raise', 'waist_bend', 'knee_bend', 'cycling', 'stairs', 'walking', 'jogging', 'running', 'jumping']
lab = lambda a: a.replace('_', ' ')
BLUE, ORANGE, GREY = '#2f7bd9', '#e8653a', '#555555'
def paper_style():
    plt.rcdefaults()
    plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 7.5, 'axes.spines.top': False, 'axes.spines.right': False,
                         'axes.edgecolor': GREY, 'xtick.color': GREY, 'ytick.color': GREY, 'axes.labelcolor': '#222222'})

In [2]:
# Fig. 1: study workflow (two rows of five stages); the same drawing is used for the paper and, larger, for the slides
STAGES = [  # (title, line 1, line 2, colour family)
    ('MHEALTH (UCI)', '10 people, 12 activities', '3 positions, 50 Hz', 'data'),
    ('Cleaning', 'drop null class (72%)', 'drop magnetometer', 'data'),
    ('Windowing', '2 s windows, 0.2 s step', '33,148 windows', 'data'),
    ('Features (RQ1)', '42 features with SQL', 'mean, SD, magnitude, trend', 'data'),
    ('Person-wise splits', 'leave-one-person-out', 'ROAR: 5 splits of people', 'split'),
    ('Ten classifiers (RQ2)', '4 families + naive baseline', 'Wilcoxon + TOST (±0.03)', 'model'),
    ('Five explainers', 'TreeSHAP, KernelSHAP, LIME', 'permutation, split gain', 'rq3'),
    ('Retraining check (RQ3)', 'masking vs ROAR', 'LightGBM + Extra Trees', 'rq3'),
    ('Position choice (RQ3)', 'drop or keep one position', '3 models × 3 seeds, MI', 'rq3'),
    ('External check', 'PAMAP2, 8 people', 'same positions and models', 'ext')]
COL = {'data': ('#eef4fc', '#2f6fbf'), 'split': ('#edf7ec', '#3f8f3c'), 'model': ('#fdf1e7', '#d9792b'),
       'rq3': ('#f3eefa', '#7b52b8'), 'ext': ('#e9f6f5', '#23897f')}
LEG = [('data', 'Data and features (RQ1)'), ('split', 'Evaluation design'), ('model', 'Classifier comparison (RQ2)'),
       ('rq3', 'Explanations and positions (RQ3)'), ('ext', 'External dataset')]
def draw_workflow(path, width, title_pt, body_pt, dpi, font='Carlito'):
    plt.rcdefaults(); plt.rcParams.update({'font.family': font})
    W, Hc = 100, 36.5
    fig = plt.figure(figsize=(width, width * Hc / W)); ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, W); ax.set_ylim(0, Hc); ax.axis('off')
    bw, bh, hh, gap, x0 = 17.5, 12.6, 4.0, 2.3, 0.9
    xs = [x0 + i * (bw + gap) for i in range(5)]
    ytop, ybot = 20.6, 4.6
    pos = [(xs[i], ytop) for i in range(5)] + [(xs[4 - i], ybot) for i in range(5)]
    lw = max(0.6, width / 10); r = 0.8
    for n, ((title, l1, l2, fam), (x, y)) in enumerate(zip(STAGES, pos), 1):
        face, edge = COL[fam]
        ax.add_patch(FancyBboxPatch((x + 0.25, y - 0.35), bw, bh, boxstyle=f'round,pad=0,rounding_size={r}', fc='#000000', ec='none', alpha=0.07, zorder=1))
        ax.add_patch(FancyBboxPatch((x, y), bw, bh, boxstyle=f'round,pad=0,rounding_size={r}', fc=face, ec=edge, lw=lw, zorder=2))
        # coloured header: rounded box clipped to its top part
        hdr = FancyBboxPatch((x, y), bw, bh, boxstyle=f'round,pad=0,rounding_size={r}', fc=edge, ec='none', zorder=3)
        ax.add_patch(hdr)
        clip = plt.Rectangle((x - 1, y + bh - hh), bw + 2, hh + 1, transform=ax.transData); hdr.set_clip_path(clip)
        cy = y + bh - hh / 2
        ax.add_patch(Circle((x + 1.75, cy), 1.25, fc='white', ec='none', zorder=4))
        ax.text(x + 1.75, cy - 0.05, str(n), ha='center', va='center', fontsize=title_pt * 0.85, fontweight='bold', color=edge, zorder=5)
        ax.text(x + 3.5, cy - 0.05, title, ha='left', va='center', fontsize=title_pt, fontweight='bold', color='white', zorder=5)
        for k, t in enumerate((l1, l2)):
            yy = y + bh - hh - 2.6 - k * 3.2
            ax.add_patch(Circle((x + 1.5, yy), 0.32, fc=edge, ec='none', zorder=4))
            ax.text(x + 2.4, yy - 0.05, t, ha='left', va='center', fontsize=body_pt, color='#2b2b2b', zorder=5)
    ac = '#8a8f98'
    def arrow(x1, x2, y):
        ax.add_patch(FancyArrowPatch((x1, y), (x2, y), arrowstyle='-|>', mutation_scale=4 + width * 0.9, lw=lw * 1.3, color=ac, shrinkA=0, shrinkB=0, zorder=1))
    for i in range(4):
        arrow(xs[i] + bw + 0.3, xs[i + 1] - 0.3, ytop + bh / 2 - hh / 2 + 0.3)
        arrow(xs[4 - i] - 0.3, xs[3 - i] + bw + 0.3, ybot + bh / 2 - hh / 2 + 0.3)
    xr = xs[4] + bw; ya, yb = ytop + bh / 2, ybot + bh / 2; xo = xr + 1.0
    verts = [(xr + 0.2, ya), (xo, ya), (xo, yb), (xr + 0.3, yb)]
    ax.add_patch(FancyArrowPatch(path=Path(verts, [1, 2, 2, 2]),
                                 arrowstyle='-|>', mutation_scale=4 + width * 0.9, lw=lw * 1.3, color=ac, joinstyle='miter', capstyle='round', zorder=1))
    # legend row
    lx = x0; ly = 1.6
    for fam, name in LEG:
        face, edge = COL[fam]
        ax.add_patch(FancyBboxPatch((lx, ly - 0.75), 2.2, 1.5, boxstyle='round,pad=0,rounding_size=0.35', fc=edge, ec='none'))
        t = ax.text(lx + 2.9, ly - 0.05, name, ha='left', va='center', fontsize=body_pt * 0.92, color='#444444')
        fig.canvas.draw(); bb = t.get_window_extent().transformed(ax.transData.inverted())
        lx = bb.x1 + 2.6
    fig.savefig(path, dpi=dpi, facecolor='white'); plt.close(fig)
draw_workflow('report/paper/fig1_workflow.png', width=7.16, title_pt=7.2, body_pt=6.1, dpi=300)
draw_workflow('report/slides/slide_workflow.png', width=18.8, title_pt=18.5, body_pt=16, dpi=150)
print('fig1 done')

fig1 done


In [3]:
# Fig. 2-4 (RQ1): windows per activity, motion per position, variation between people
paper_style()
n = R('table_q5.csv').set_index('activity')['n'].reindex(ACT)
fig, ax = plt.subplots(figsize=(3.4, 2.3))
ax.barh([lab(a) for a in ACT][::-1], n.values[::-1], color=[ORANGE if v == n.min() else BLUE for v in n.values[::-1]], height=0.72)
for i, v in enumerate(n.values[::-1]): ax.text(v + 25, i, f'{v:,}', va='center', fontsize=7, color=GREY)
ax.set_xlim(0, 3500); ax.set_xlabel('Number of 2 s windows (10 people)'); ax.grid(axis='x', color='#e5e5e5', lw=0.5); ax.set_axisbelow(True)
fig.tight_layout(); fig.savefig('report/paper/fig2_windows_per_activity.png', dpi=300); plt.close(fig)

def heat(M, fmt, cmap, label, path, norm=None, vmin=None, vmax=None):
    paper_style()
    fig, ax = plt.subplots(figsize=(3.4, 2.7))
    im = ax.imshow(M, cmap=cmap, norm=norm, vmin=vmin, vmax=vmax, aspect='auto')
    ax.set_xticks(range(3), ['Chest', 'Ankle', 'Wrist']); ax.xaxis.tick_top(); ax.tick_params(length=0)
    ax.set_yticks(range(len(ACT)), [lab(a) for a in ACT])
    for s in ax.spines.values(): s.set_visible(False)
    hi = (np.log10(M.max()) + np.log10(M.min())) / 2 if norm is not None else (M.max() + M.min()) / 2
    for i in range(M.shape[0]):
        for j in range(3):
            v = M[i, j]; dark = (np.log10(v) if norm is not None else v) > hi
            ax.text(j, i, fmt(v), ha='center', va='center', fontsize=6.5, color='white' if dark else '#111111')
    cb = fig.colorbar(im, ax=ax, fraction=0.05, pad=0.03); cb.set_label(label, fontsize=6.5); cb.ax.tick_params(labelsize=6)
    fig.tight_layout(); fig.savefig(path, dpi=300); plt.close(fig)
q2 = R('table_q2.csv').set_index('activity').reindex(ACT)
heat(q2[['chest_motion', 'ankle_motion', 'wrist_motion']].values, lambda v: f'{v:.2f}', 'Blues',
     'SD of acceleration magnitude (m/s², log scale)', 'report/paper/fig3_motion_heatmap.png', norm=LogNorm(0.08, 12))
q3 = R('table_q3.csv').set_index('activity').reindex(ACT)
heat(q3[['chest_cv', 'ankle_cv', 'wrist_cv']].values * 100, lambda v: f'{v:.0f}%', 'Oranges',
     'Coefficient of variation between people (%)', 'report/paper/fig4_people_cv.png', vmin=0, vmax=40)
print(q2.round(2).iloc[:, 1:4].to_string()); print((q3[['chest_cv', 'ankle_cv', 'wrist_cv']] * 100).round(0).to_string())

            chest_motion  ankle_motion  wrist_motion
activity                                            
standing            0.15          0.10          0.11
lying               0.14          0.10          0.12
sitting             0.16          0.11          0.10
arm_raise           0.31          0.11          2.75
waist_bend          1.18          0.24          1.99
knee_bend           1.91          0.58          2.18
cycling             1.21          1.97          2.08
stairs              2.03          3.30          1.99
walking             2.20          5.20          2.21
jogging             8.08          6.25          7.50
running             8.58          7.01          6.70
jumping             9.23          7.87          9.18
            chest_cv  ankle_cv  wrist_cv
activity                                
standing        14.0       3.0      12.0
lying           16.0       6.0      15.0
sitting         15.0       6.0       7.0
arm_raise       21.0       5.0      27.0
waist_bend  

In [4]:
# Fig. 5: masking vs retraining on the main split (LightGBM, top 10 features, mean over three seeds)
paper_style(); plt.rcParams['font.size'] = 9
r = R('table_rq3_roar.csv'); r = r[r.k == 10].set_index('method')
ORDER = ['TreeSHAP', 'KernelSHAP', 'LIME', 'Permutation', 'Gain', 'Random']; NAME = {'Gain': 'Split gain'}
base = 0.947   # all 42 features, mean over three seeds (table_rq3_roar_splits.csv, split 0, LightGBM)
def masking_plot(path, size, fs, dot, dpi, xlabel):
    fig, ax = plt.subplots(figsize=size); y = np.arange(len(ORDER))[::-1]
    for yy, m in zip(y, ORDER):
        a, b = r.loc[m, 'deleted'], r.loc[m, 'retrained']
        ax.hlines(yy, a, b, color='#d5d8dc', lw=dot / 12, zorder=1)
        ax.scatter(a, yy, s=dot, color=ORANGE, zorder=3, edgecolor='white', linewidth=0.6)
        ax.scatter(b, yy, s=dot, color=BLUE, zorder=3, edgecolor='white', linewidth=0.6)
        if m != 'Random': ax.text(a - 0.02, yy, f'{a:.2f}', ha='right', va='center', fontsize=fs * 0.9, color='#444444')
        ax.text(1.03, yy, f'{b:.2f}', ha='left', va='center', fontsize=fs * 0.9, color='#444444')
    ax.axvline(base, ls='--', color='#444444', lw=0.9)
    ax.text(base - 0.01, len(ORDER) - 0.55, f'all 42 features: {base:.3f}', ha='right', va='bottom', fontsize=fs * 0.85, color='#444444')
    ax.set_yticks(y, [NAME.get(m, m) for m in ORDER], fontsize=fs); ax.tick_params(axis='x', labelsize=fs)
    ax.set_xlim(0.15, 1.1); ax.set_ylim(-0.6, len(ORDER) - 0.1); ax.set_xticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.grid(axis='x', color='#e5e5e5', lw=0.6); ax.set_axisbelow(True)
    ax.scatter([], [], color=ORANGE, s=dot, label='Masked, not retrained'); ax.scatter([], [], color=BLUE, s=dot, label='Removed and retrained')
    ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.0), ncol=2, frameon=False, fontsize=fs * 0.9)
    ax.set_xlabel(xlabel, fontsize=fs)
    fig.tight_layout(); fig.savefig(path, dpi=dpi, bbox_inches='tight'); plt.close(fig)
masking_plot('report/paper/fig5_masking_vs_retraining.png', (4.2, 3.0), 9, 30, 300, 'Macro-F1, people 8 to 10, top 10 removed (LightGBM)')
shutil.copy('report/fig_rq3_positions_3models.png', 'report/paper/fig6_positions_3models.png')
print(r[['deleted', 'retrained', 'retrained_sd']].round(3).to_string())

             deleted  retrained  retrained_sd
method                                       
TreeSHAP       0.368      0.930         0.002
KernelSHAP     0.464      0.923         0.005
LIME           0.368      0.956         0.001
Permutation    0.500      0.905         0.003
Gain           0.382      0.944         0.010
Random         0.937      0.951         0.001


In [5]:
# Paper Tables I-III as LaTeX, straight from the CSV files
t2 = R('table_rq2.csv').set_index('model')
order2 = ['Extra Trees', 'XGBoost', 'Logistic', 'HistGB', 'Random Forest', 'Linear SVM', 'LightGBM', 'Small MLP', 'k-NN', 'Decision Tree', 'Majority (naive)']
rows = [f'{m} & {t2.loc[m,"macroF1"]:.3f} & {t2.loc[m,"sd"]:.3f} & {t2.loc[m,"worst_person"]:.3f} & {t2.loc[m,"accuracy"]:.3f} & {t2.loc[m,"ms_per_window"]:.3f} \\\\' for m in order2]
share = R('table_rq3_group_share.csv').set_index(R('table_rq3_group_share.csv').columns[0])
meth = R('table_rq3_methods.csv'); meth = meth.set_index(meth.columns[0])
print(meth.round(3).to_string())
def col(df, *names):
    for n in names:
        if n in df.columns: return df[n]
    raise KeyError(names)
stab, secs = col(meth, 'stability', 'rank_stability', 'spearman'), col(meth, 'seconds', 'time_s')
rows3 = []
for m, name in [('TreeSHAP', 'TreeSHAP'), ('KernelSHAP', 'KernelSHAP'), ('LIME', 'LIME'), ('Permutation', 'Permutation'), ('Gain', 'Split gain')]:
    s = share[m]; t = secs[m]
    rows3.append(f'{name} & {s["ankle"]:.3f} & {s["wrist"]:.3f} & {s["chest"]:.3f} & {s["heart signal"]:.3f} & {stab[m]:.2f} & '
                 + ('$<$0.01' if t < 0.01 else f'{t:.1f}') + ' \\\\')
ext = R('table_external_check.csv')
rows4, last = [], None
for _, x in ext.iterrows():
    cells = [x['chest device'], x['wrist'], x['ankle']]; best = int(np.argmin(cells))
    c = [f'\\textbf{{{v:.3f}}}' if i == best else f'{v:.3f}' for i, v in enumerate(cells)]
    ds = x['data set'] if x['data set'] != last else ''
    if last is not None and ds: rows4.append('\\midrule')
    rows4.append(f'{ds} & {x["model"]} & ' + ' & '.join(c) + ' \\\\'); last = x['data set']
tex = ('% Table I (RQ2): leave-one-person-out, mean over 10 people\n' + '\n'.join(rows) +
       '\n\n% Table II: share of attribution per position, rank stability, run time\n' + '\n'.join(rows3) +
       '\n\n% Table III: macro-F1 lost when one position is dropped\n' + '\n'.join(rows4) + '\n')
open('report/paper/tables_paper.tex', 'w').write(tex); print(tex)

             rank_stability  seconds
Unnamed: 0                          
TreeSHAP              0.542    2.834
KernelSHAP            0.565   33.966
LIME                  0.465    2.965
Permutation           0.332   44.426
Gain                  0.572    0.001
% Table I (RQ2): leave-one-person-out, mean over 10 people
Extra Trees & 0.962 & 0.048 & 0.873 & 0.967 & 0.028 \\
XGBoost & 0.958 & 0.050 & 0.869 & 0.961 & 0.012 \\
Logistic & 0.955 & 0.042 & 0.884 & 0.963 & 0.001 \\
HistGB & 0.951 & 0.049 & 0.878 & 0.955 & 0.019 \\
Random Forest & 0.950 & 0.056 & 0.846 & 0.954 & 0.029 \\
Linear SVM & 0.943 & 0.068 & 0.794 & 0.952 & 0.001 \\
LightGBM & 0.933 & 0.053 & 0.855 & 0.939 & 0.047 \\
Small MLP & 0.927 & 0.047 & 0.871 & 0.940 & 0.002 \\
k-NN & 0.912 & 0.060 & 0.838 & 0.929 & 0.076 \\
Decision Tree & 0.893 & 0.072 & 0.786 & 0.906 & 0.001 \\
Majority (naive) & 0.014 & 0.000 & 0.013 & 0.090 & 0.000 \\

% Table II: share of attribution per position, rank stability, run time
TreeSHAP & 0.399 & 0

In [6]:
# Slide versions with large fonts (pictures meant to fill most of a 16:9 slide)
plt.rcdefaults()
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 44, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.edgecolor': '#4a5168', 'axes.labelcolor': '#4a5168', 'xtick.color': '#4a5168', 'ytick.color': '#4a5168'})
NAVY = '#2b3a8c'
masking_plot('report/slides/slide_masking_vs_retraining.png', (15, 8.1), 44, 800, 120, 'Score after removing the top 10 features')
ext = R('table_external_check.csv'); cols = ['chest device', 'wrist', 'ankle']
for ds in ['MHEALTH', 'PAMAP2']:
    d = ext[ext['data set'] == ds].set_index('model')
    fig, ax = plt.subplots(figsize=(15, 7.6)); x = np.arange(3); w = 0.27
    for j, (m, c) in enumerate(zip(['LightGBM', 'Extra Trees', 'XGBoost'], [NAVY, BLUE, '#8fa8e8'])):
        ax.bar(x + (j - 1) * w, d.loc[m, cols].values.astype(float), w * 0.92, color=c, label=m)
    for i, cname in enumerate(cols):
        v = d[cname].values; lo, hi = v.min(), v.max()
        s = f'{lo:.2f}' if round(lo, 2) == round(hi, 2) else f'{lo:.2f}–{hi:.2f}'
        ax.text(i, hi + 0.004, s, ha='center', va='bottom', fontsize=44, color=ORANGE if i == 0 else '#4a5168', fontweight='bold')
    ax.set_xticks(x, ['Drop chest', 'Drop wrist', 'Drop ankle']); ax.get_xticklabels()[0].set_color(ORANGE); ax.get_xticklabels()[0].set_fontweight('bold')
    ax.set_ylabel('Score lost'); ax.set_ylim(0, 0.135); ax.set_yticks([0, 0.05, 0.10])
    ax.legend(frameon=False, fontsize=40, loc='upper left', ncol=3, bbox_to_anchor=(-0.02, 1.22), handlelength=1, columnspacing=1)
    ax.grid(axis='y', color='#dde3f3', lw=1.5); ax.set_axisbelow(True)
    fig.savefig(f'report/slides/slide_drop_cost_{ds.lower()}.png', dpi=120, bbox_inches='tight', pad_inches=0.2); plt.close(fig)
t2 = R('table_rq2.csv'); t2 = t2[t2.model != 'Majority (naive)'].sort_values('macroF1')
fig, ax = plt.subplots(figsize=(15, 8.5))
ax.barh(t2.model, t2.macroF1, color=[BLUE if v >= t2.macroF1.nlargest(5).min() else '#b9c7e8' for v in t2.macroF1], height=0.7)
for i, v in enumerate(t2.macroF1): ax.text(v + 0.003, i, f'{v:.3f}', va='center', fontsize=36, color='#4a5168')
ax.set_xlim(0.85, 1.0); ax.set_xticks([0.85, 0.90, 0.95, 1.0]); ax.set_xlabel('Macro-F1 on an unseen person')
fig.savefig('report/slides/slide_rq2_models.png', dpi=120, bbox_inches='tight', pad_inches=0.2); plt.close(fig)
print(sorted(os.listdir('report/slides')))

['slide_drop_cost_mhealth.png', 'slide_drop_cost_pamap2.png', 'slide_masking_vs_retraining.png', 'slide_rq2_models.png', 'slide_workflow.png']


In [7]:
# Checks
need = ['fig1_workflow.png', 'fig2_windows_per_activity.png', 'fig3_motion_heatmap.png', 'fig4_people_cv.png',
        'fig5_masking_vs_retraining.png', 'fig6_positions_3models.png', 'tables_paper.tex']
assert all(os.path.exists('report/paper/' + f) for f in need)
assert len([f for f in os.listdir('report/slides') if f.endswith('.png')]) == 5
print('all figures and tables written')

all figures and tables written
